# LLM-Powered Chatbot System — Reference Implementation

This notebook is a runnable Google Colab implementation of the chatbot system described in the accompanying report ("System Requirement & Implementation" — SRII). It mirrors the architecture from **Section 4 (System Design)**:

- **4.1 System Architecture** → `Chatbot` orchestrator class ties every layer together
- **4.5 Database Design** → `SessionStore` (USER/SESSION/CHAT_MESSAGE/FEEDBACK) + `KnowledgeBase` (KNOWLEDGE_DOC)
- **4.6 Module Description** → one class per module below (Auth, Session Manager, Retrieval, Prompt Builder, LLM Client, Post-Processor, Logger)
- **4.7 Process Flow** → `Chatbot.chat()` implements the flowchart step by step
- **4.9 Security** → hashed passwords, server-side API key, stateless orchestrator, content-safety filter

Run the cells top to bottom. **No API key required** — the LLM Inference Client below runs a small open-source instruction-tuned model locally via Hugging Face `transformers`, so everything works out of the box on Colab's free tier.

> Tip: for noticeably faster responses, go to **Runtime → Change runtime type → T4 GPU** before running. It'll work on CPU too, just slower per reply.


## 0. Setup — Install Dependencies

In [ ]:
!pip install -q transformers accelerate scikit-learn ipywidgets pandas matplotlib


In [ ]:
import uuid
import time
import hashlib
import re
from dataclasses import dataclass, field
from typing import List, Dict, Optional

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Environment ready.")


## 1. Authentication & Session Manager (Module 4.6 / FR-1)

Minimal in-memory USER and SESSION stores. Passwords are hashed (never stored in plain text — see 4.9 Security), and a short-lived session token is issued on login, mirroring the JWT-based approach described in the report.

In [ ]:
class AuthModule:
    """Handles user registration/login and session token issuance."""

    def __init__(self):
        self._users: Dict[str, Dict] = {}   # email -> {password_hash, user_id}
        self._tokens: Dict[str, str] = {}   # token -> email

    @staticmethod
    def _hash_password(password: str, salt: str = "srii_salt") -> str:
        return hashlib.sha256((salt + password).encode()).hexdigest()

    def register(self, email: str, password: str) -> str:
        if email in self._users:
            raise ValueError("User already exists.")
        user_id = str(uuid.uuid4())
        self._users[email] = {"password_hash": self._hash_password(password), "user_id": user_id}
        return user_id

    def login(self, email: str, password: str) -> str:
        user = self._users.get(email)
        if not user or user["password_hash"] != self._hash_password(password):
            raise PermissionError("Invalid credentials.")
        token = str(uuid.uuid4())
        self._tokens[token] = email
        return token

    def authenticate(self, token: str) -> Optional[str]:
        """Returns the user's email if the token is valid, else None."""
        return self._tokens.get(token)


auth = AuthModule()
demo_user_id = auth.register("demo@example.com", "password123")
demo_token = auth.login("demo@example.com", "password123")
print("Registered demo user:", demo_user_id)
print("Session token:", demo_token)


## 2. Database Design — Session & Chat Store (Section 4.5, ER Diagram)

Implements USER → SESSION → CHAT_MESSAGE → FEEDBACK as in-memory tables (a real deployment would back this with Postgres/SQLite as described in 4.8 Technology Stack). Foreign keys are modelled as dictionary references, and lookups are indexed by session id, matching the indexing note in 4.5.

In [ ]:
@dataclass
class ChatMessage:
    message_id: str
    session_id: str
    role: str            # "user" or "assistant"
    content: str
    timestamp: float
    feedback: Optional[Dict] = None


class SessionStore:
    """USER has many SESSIONs; SESSION has many CHAT_MESSAGEs (Fig 4.6 ER diagram)."""

    def __init__(self):
        self.sessions: Dict[str, Dict] = {}          # session_id -> {user_id, created_at}
        self.messages: Dict[str, List[ChatMessage]] = {}   # session_id -> [ChatMessage]

    def create_session(self, user_id: str) -> str:
        session_id = str(uuid.uuid4())
        self.sessions[session_id] = {"user_id": user_id, "created_at": time.time()}
        self.messages[session_id] = []
        return session_id

    def add_message(self, session_id: str, role: str, content: str) -> ChatMessage:
        msg = ChatMessage(
            message_id=str(uuid.uuid4()),
            session_id=session_id,
            role=role,
            content=content,
            timestamp=time.time(),
        )
        self.messages[session_id].append(msg)
        return msg

    def get_history(self, session_id: str, max_turns: int = 10) -> List[ChatMessage]:
        """Context-window management (Section 2, 'Context management' challenge)."""
        return self.messages.get(session_id, [])[-max_turns * 2:]

    def add_feedback(self, session_id: str, message_id: str, rating: int, comment: str = ""):
        for msg in self.messages.get(session_id, []):
            if msg.message_id == message_id:
                msg.feedback = {"rating": rating, "comment": comment}
                return True
        return False


session_store = SessionStore()
demo_session_id = session_store.create_session(demo_user_id)
print("Created session:", demo_session_id)


## 3. Retrieval Module — Knowledge Base (Section 4.1 / 4.6, RAG)

A lightweight TF-IDF vector store standing in for the "vector database or knowledge base" described in the architecture. In production this would be a proper embedding-based vector DB (e.g. pgvector, Pinecone); TF-IDF keeps the notebook self-contained and dependency-light while demonstrating the same retrieval-augmented generation pattern.

In [ ]:
class KnowledgeBase:
    """KNOWLEDGE_DOC store + retrieval, queried at runtime by the orchestrator."""

    def __init__(self):
        self.docs: List[str] = []
        self.doc_ids: List[str] = []
        self._vectorizer: Optional[TfidfVectorizer] = None
        self._matrix = None

    def add_document(self, text: str, doc_id: Optional[str] = None) -> str:
        doc_id = doc_id or str(uuid.uuid4())
        self.docs.append(text)
        self.doc_ids.append(doc_id)
        self._rebuild_index()
        return doc_id

    def _rebuild_index(self):
        if not self.docs:
            return
        self._vectorizer = TfidfVectorizer(stop_words="english")
        self._matrix = self._vectorizer.fit_transform(self.docs)

    def retrieve(self, query: str, top_k: int = 2, min_score: float = 0.05) -> List[str]:
        """Returns up to top_k relevant snippets, or [] if nothing clears min_score
        (mirrors the Fig 4.7 decision point: retrieval only fires when the query needs it)."""
        if not self.docs or self._vectorizer is None:
            return []
        query_vec = self._vectorizer.transform([query])
        scores = cosine_similarity(query_vec, self._matrix).flatten()
        ranked = sorted(zip(scores, self.docs), key=lambda x: x[0], reverse=True)
        return [doc for score, doc in ranked[:top_k] if score >= min_score]


knowledge_base = KnowledgeBase()

# --- Seed with sample domain knowledge (swap this for your own KB) ---
sample_docs = [
    "Our support hours are Monday to Friday, 9am to 6pm IST. Response time is under 2 hours.",
    "Refunds are processed within 5-7 business days back to the original payment method.",
    "The Pro plan costs $20/month and includes unlimited chatbot sessions and priority support.",
    "To reset your password, go to Settings > Security > Reset Password and follow the emailed link.",
]
for d in sample_docs:
    knowledge_base.add_document(d)

print(f"Knowledge base loaded with {len(knowledge_base.docs)} documents.")
print("Test retrieval:", knowledge_base.retrieve("How much does the Pro plan cost?"))


## 4. Prompt Builder (Section 4.1, Conversation Orchestrator)

Combines the system instructions, recent conversation history, and any retrieved knowledge-base context into the final prompt sent to the LLM.

In [ ]:
SYSTEM_PROMPT = (
    "You are a helpful, concise customer-support assistant. "
    "Answer using the conversation history and any provided context. "
    "If the context does not contain the answer, say you are not sure rather than guessing."
)


class PromptBuilder:
    @staticmethod
    def build_messages(history: List[ChatMessage], retrieved_context: List[str]) -> List[Dict]:
        messages = []
        if retrieved_context:
            context_block = "\n\n".join(f"- {c}" for c in retrieved_context)
            messages.append({
                "role": "user",
                "content": f"[Reference context, not visible to the user directly]\n{context_block}",
            })
            messages.append({"role": "assistant", "content": "Understood, I will use this context if relevant."})

        for m in history:
            messages.append({"role": m.role, "content": m.content})

        return messages


## 5. LLM Inference Client — Local Open-Source Model (Section 4.1 / 4.8)

No API key needed. This loads a small, free instruction-tuned model (`Qwen2.5-0.5B-Instruct`) directly from Hugging Face and runs it locally in the Colab runtime — swapping out the "LLM Inference Layer" box in the architecture diagram for a self-hosted model instead of a paid API. The rest of the system (retrieval, prompting, safety filtering, logging) is unchanged, which is exactly the point of the layered design in 4.8: the model/vendor is decoupled behind this one class.

The first run downloads the model (~1GB) — this takes a minute or two, then responses are fast.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

DEFAULT_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"   # small, free, instruction-tuned; swappable per 4.8

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading {DEFAULT_MODEL} on {device} ... (first run downloads ~1GB, please wait)")

tokenizer = AutoTokenizer.from_pretrained(DEFAULT_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    DEFAULT_MODEL,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)


class LLMClient:
    """Local inference client — same interface as an API-backed client would have,
    so it can be swapped for Claude, GPT, etc. without touching the orchestrator."""

    def __init__(self, model, tokenizer, device: str, model_name: str = DEFAULT_MODEL):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        self.model_name = model_name

    def generate(self, system_prompt: str, messages: List[Dict], max_tokens: int = 300, temperature: float = 0.7) -> str:
        chat = [{"role": "system", "content": system_prompt}] + messages
        prompt_text = self.tokenizer.apply_chat_template(
            chat, tokenize=False, add_generation_prompt=True
        )
        inputs = self.tokenizer(prompt_text, return_tensors="pt").to(self.device)

        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                temperature=temperature,
                do_sample=temperature > 0,
                pad_token_id=self.tokenizer.eos_token_id,
            )

        generated = output_ids[0][inputs["input_ids"].shape[1]:]
        return self.tokenizer.decode(generated, skip_special_tokens=True).strip()


llm_client = LLMClient(model, tokenizer, device)
print("LLM client ready, using model:", llm_client.model_name, "on", device)


## 6. Response Post-Processor (Section 2 & 4.9, Safety Filtering)

A lightweight keyword-based content-safety check plus formatting cleanup, standing in for the "Response Post-Processor" module in the architecture diagram. Swap in a moderation API call for production use.

In [ ]:
BLOCKED_PATTERNS = [
    r"\bhow to (make|build) (a )?(bomb|weapon)\b",
    r"\bself[- ]harm\b",
]

FALLBACK_MESSAGE = "I can't help with that request. Is there something else I can assist you with?"


class ResponsePostProcessor:
    @staticmethod
    def is_safe(text: str) -> bool:
        lowered = text.lower()
        return not any(re.search(pattern, lowered) for pattern in BLOCKED_PATTERNS)

    @staticmethod
    def process(raw_response: str, user_message: str) -> str:
        if not ResponsePostProcessor.is_safe(user_message) or not ResponsePostProcessor.is_safe(raw_response):
            return FALLBACK_MESSAGE
        return raw_response.strip()


## 7. Logging, Analytics & Feedback Store (Section 4.6 / FR-7)

Every turn is logged to a pandas DataFrame acting as the analytics store described in the report. `add_feedback` closes the feedback loop (five-star rating + comment).

In [ ]:
class AnalyticsLogger:
    def __init__(self):
        self.records: List[Dict] = []

    def log_turn(self, session_id: str, user_message: str, assistant_message: str,
                 used_retrieval: bool, latency_ms: float, message_id: str):
        self.records.append({
            "session_id": session_id,
            "message_id": message_id,
            "user_message": user_message,
            "assistant_message": assistant_message,
            "used_retrieval": used_retrieval,
            "latency_ms": round(latency_ms, 1),
            "timestamp": time.time(),
            "rating": None,
        })

    def add_feedback(self, message_id: str, rating: int, comment: str = ""):
        for r in self.records:
            if r["message_id"] == message_id:
                r["rating"] = rating
                r["comment"] = comment
                return True
        return False

    def as_dataframe(self) -> pd.DataFrame:
        return pd.DataFrame(self.records)

    def summary(self) -> Dict:
        df = self.as_dataframe()
        if df.empty:
            return {"turns": 0}
        return {
            "turns": len(df),
            "avg_latency_ms": round(df["latency_ms"].mean(), 1),
            "retrieval_usage_pct": round(100 * df["used_retrieval"].mean(), 1),
            "avg_rating": round(df["rating"].dropna().mean(), 2) if df["rating"].notna().any() else None,
        }


analytics = AnalyticsLogger()


## 8. Conversation Orchestrator — the `Chatbot` class (Section 4.1 / 4.7 Process Flow)

Ties every module together, implementing the message-processing flowchart:
authenticate → build history → retrieve (if useful) → build prompt → call LLM → post-process → log → return.

In [ ]:
class Chatbot:
    def __init__(self, auth: AuthModule, sessions: SessionStore, kb: KnowledgeBase,
                 llm: LLMClient, logger: AnalyticsLogger):
        self.auth = auth
        self.sessions = sessions
        self.kb = kb
        self.llm = llm
        self.logger = logger

    def chat(self, token: str, session_id: str, user_message: str) -> Dict:
        # 1. Authenticate (FR-1)
        email = self.auth.authenticate(token)
        if not email:
            raise PermissionError("Invalid or expired session token.")

        start = time.time()

        # 2. Persist the incoming user message
        self.sessions.add_message(session_id, "user", user_message)

        # 3. Retrieval (only when it might help — Fig 4.7 decision point)
        retrieved = self.kb.retrieve(user_message, top_k=2)

        # 4. Build prompt from history + retrieved context
        history = self.sessions.get_history(session_id)
        messages = PromptBuilder.build_messages(history, retrieved)

        # 5. LLM inference
        raw_response = self.llm.generate(SYSTEM_PROMPT, messages)

        # 6. Post-process for safety/formatting
        final_response = ResponsePostProcessor.process(raw_response, user_message)

        # 7. Persist assistant message
        assistant_msg = self.sessions.add_message(session_id, "assistant", final_response)

        # 8. Log for analytics
        latency_ms = (time.time() - start) * 1000
        self.logger.log_turn(
            session_id=session_id,
            user_message=user_message,
            assistant_message=final_response,
            used_retrieval=bool(retrieved),
            latency_ms=latency_ms,
            message_id=assistant_msg.message_id,
        )

        return {
            "response": final_response,
            "message_id": assistant_msg.message_id,
            "used_retrieval": bool(retrieved),
            "latency_ms": round(latency_ms, 1),
        }


chatbot = Chatbot(auth, session_store, knowledge_base, llm_client, analytics)
print("Chatbot orchestrator ready.")


## 9. Quick Test — Non-Interactive

In [ ]:
result = chatbot.chat(demo_token, demo_session_id, "How much does the Pro plan cost, and does it include priority support?")
print("Assistant:", result["response"])
print("\nUsed retrieval:", result["used_retrieval"], "| Latency:", result["latency_ms"], "ms")


## 10. Interactive Chat UI (Section 5.3 / 5.7 — Chat Window + Feedback Panel)

An `ipywidgets`-based chat window with a five-star feedback control per response, mirroring Screenshots 5.3 and 5.7 from the report. Works natively in Google Colab.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

chat_log = widgets.Output(layout=widgets.Layout(border="1px solid #ddd", height="320px", overflow_y="auto", padding="8px"))
text_input = widgets.Text(placeholder="Type a message...", layout=widgets.Layout(width="75%"))
send_button = widgets.Button(description="Send", button_style="primary")
status_label = widgets.Label(value="")

last_message_id = {"id": None}

def render_feedback_row(message_id):
    stars = widgets.ToggleButtons(options=["1", "2", "3", "4", "5"], description="Rate:")
    comment_box = widgets.Text(placeholder="Optional comment", layout=widgets.Layout(width="50%"))
    submit_fb = widgets.Button(description="Submit feedback", button_style="info")
    fb_status = widgets.Label(value="")

    def on_submit(_):
        rating = int(stars.value)
        session_store.add_feedback(demo_session_id, message_id, rating, comment_box.value)
        analytics.add_feedback(message_id, rating, comment_box.value)
        fb_status.value = "Thanks for your feedback!"

    submit_fb.on_click(on_submit)
    return widgets.VBox([widgets.HBox([stars, comment_box, submit_fb]), fb_status])

def on_send(_):
    user_text = text_input.value.strip()
    if not user_text:
        return
    text_input.value = ""
    status_label.value = "Thinking..."
    with chat_log:
        print(f"You: {user_text}")
    try:
        result = chatbot.chat(demo_token, demo_session_id, user_text)
        with chat_log:
            print(f"Assistant: {result['response']}")
            if result["used_retrieval"]:
                print("  (used knowledge-base context)")
            print()
        last_message_id["id"] = result["message_id"]
        display(render_feedback_row(result["message_id"]))
    except Exception as e:
        with chat_log:
            print(f"[Error] {e}")
    status_label.value = ""

send_button.on_click(on_send)
text_input.on_submit(on_send)

display(widgets.VBox([
    chat_log,
    widgets.HBox([text_input, send_button]),
    status_label,
]))


## 11. Analytics Dashboard (Screenshot 5.6-style admin view)

In [ ]:
summary = analytics.summary()
print("Session analytics summary:")
for k, v in summary.items():
    print(f"  {k}: {v}")

df = analytics.as_dataframe()
df


In [ ]:
import matplotlib.pyplot as plt

if not df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    axes[0].plot(range(len(df)), df["latency_ms"], marker="o")
    axes[0].set_title("Response Latency per Turn")
    axes[0].set_xlabel("Turn")
    axes[0].set_ylabel("Latency (ms)")

    retrieval_counts = df["used_retrieval"].value_counts()
    axes[1].bar(retrieval_counts.index.astype(str), retrieval_counts.values, color=["#888", "#4C72B0"])
    axes[1].set_title("Retrieval Usage")
    axes[1].set_xlabel("Used retrieval?")
    axes[1].set_ylabel("Turn count")

    plt.tight_layout()
    plt.show()
else:
    print("No turns logged yet — chat with the bot above first.")


## 12. Extending This Notebook

- **Bigger knowledge base**: replace `KnowledgeBase` (TF-IDF) with a proper vector DB (e.g. `chromadb`, `pgvector`, or Pinecone) and an embeddings model for semantic (not just keyword) retrieval — Section 6.1 Future Scope.
- **Persistence**: swap the in-memory `SessionStore` / `AnalyticsLogger` for SQLite/Postgres so data survives notebook restarts.
- **Multi-user**: the `AuthModule` already supports multiple registered users — add a login form widget to let different users authenticate.
- **Bigger/better local model**: swap `DEFAULT_MODEL` for a larger open model (e.g. `Qwen/Qwen2.5-3B-Instruct`, `microsoft/Phi-3-mini-4k-instruct`) if you have GPU headroom, for noticeably better answer quality.
- **Switch to a hosted API**: if you later want Claude/GPT-quality responses, just rewrite `LLMClient.generate` to call that provider's API instead — the rest of the system (retrieval, sessions, safety, logging) doesn't need to change, which is the point of decoupling this module (4.8).
- **Streaming responses**: use `model.generate(..., streamer=...)` (a `TextIteratorStreamer`) in `LLMClient.generate` for token-by-token output, matching Screenshot 5.3's "responses stream into view" behaviour.
- **Multilingual / voice**: hook a speech-to-text/text-to-speech library into the chat UI cell, per Section 6.1.
